# Pipeline 4: Reintegration Causal Analysis

**New Dawn Safehouse Management System — ML Pipeline**

---

## Table of Contents
1. Business Understanding & Problem Definition
2. Data Understanding & Exploration
3. Data Preparation
4. Multicollinearity Analysis — VIF (Ch. 9)
5. Causal Modelling — Logistic Regression (Ch. 6–9)
6. Ensemble Validation (Ch. 10)
7. Odds Ratio Interpretation (Ch. 12–13)
8. Deployment — CSV Output & Web Integration (Ch. 15)

## 1. Business Understanding & Problem Definition

New Dawn's case managers need to understand **what factors cause successful reintegration**. Unlike predictive models that optimize accuracy, this pipeline uses **causal inference** — specifically logistic regression with odds ratios — to identify modifiable factors that improve outcomes.

### Goal
Fit an interpretable logistic regression model on `reintegration_status` (binary: Completed vs. not), compute odds ratios, and generate plain-language explanations for each significant factor.

### Target
- `reintegration_completed` = 1 if `reintegration_status == 'Completed'`, else 0
- Dataset: 60 residents (19 completed, 41 not completed)

### Success Criteria
- Identify statistically significant factors (p < 0.05)
- Validate causal findings against ensemble model direction
- Produce actionable plain-language recommendations for case workers

## 2. Data Understanding & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

residents = pd.read_csv('../lighthouse_csv_v7/residents.csv')

print(f'Residents: {residents.shape}')
print(f'\nReintegration status distribution:')
print(residents['reintegration_status'].value_counts())

In [ ]:
# Reintegration outcome by key factors
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

residents['reintegration_completed'] = (residents['reintegration_status'] == 'Completed').astype(int)

# By case status
ct1 = pd.crosstab(residents['case_status'], residents['reintegration_completed'], normalize='index')
ct1[1].sort_values().plot.barh(ax=axes[0], color='#91B191')
axes[0].set_title('Reintegration Rate by Case Status')
axes[0].set_xlabel('Proportion Completed')

# By initial risk level
risk_order = ['Low', 'Medium', 'High', 'Critical']
ct2 = pd.crosstab(residents['initial_risk_level'], residents['reintegration_completed'], normalize='index')
ct2 = ct2.reindex(risk_order).dropna()
ct2[1].plot.barh(ax=axes[1], color='#A2C9E1')
axes[1].set_title('Reintegration Rate by Initial Risk')
axes[1].set_xlabel('Proportion Completed')

# By referral source
ct3 = pd.crosstab(residents['referral_source'], residents['reintegration_completed'], normalize='index')
ct3[1].sort_values().plot.barh(ax=axes[2], color='#FFCC66')
axes[2].set_title('Reintegration Rate by Referral Source')
axes[2].set_xlabel('Proportion Completed')

plt.tight_layout()
plt.show()

## 3. Data Preparation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from pipelines.reintegration_causal import prepare_data, build_feature_matrix, RISK_MAP

prepared = prepare_data(residents)
X, y, feature_names = build_feature_matrix(prepared)

print(f'Feature matrix: {X.shape}')
print(f'Target balance: {y.sum():.0f} completed / {len(y) - y.sum():.0f} not completed')
print(f'\nFeatures ({len(feature_names)}):')
for f in feature_names:
    print(f'  {f}')

## 4. Multicollinearity Analysis — VIF (Ch. 9)

High VIF (> 10) indicates severe multicollinearity that inflates coefficient standard errors and makes causal claims unreliable. We iteratively drop the highest-VIF feature.

In [ ]:
from functions import calculate_vif

vif_df = calculate_vif(X, feature_names)
print('VIF values (before dropping):')
print(vif_df.to_string())

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
vif_sorted = vif_df.sort_values('VIF', ascending=True)
colors = ['#FF6B6B' if v > 10 else '#91B191' for v in vif_sorted['VIF']]
ax.barh(vif_sorted['Feature'], vif_sorted['VIF'], color=colors)
ax.axvline(x=10, color='red', linestyle='--', label='VIF threshold (10)')
ax.set_title('Variance Inflation Factors')
ax.set_xlabel('VIF')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Iteratively drop features with VIF > 10
X_clean = X.copy()
clean_features = list(feature_names)

while True:
    vif_check = calculate_vif(X_clean, clean_features)
    max_vif = vif_check['VIF'].max()
    if max_vif <= 10:
        break
    worst = vif_check.loc[vif_check['VIF'].idxmax(), 'Feature']
    idx = clean_features.index(worst)
    X_clean = np.delete(X_clean, idx, axis=1)
    clean_features.pop(idx)
    print(f'Dropped {worst} (VIF={max_vif:.1f})')

print(f'\nRemaining features: {len(clean_features)}')
print(f'Final max VIF: {calculate_vif(X_clean, clean_features)["VIF"].max():.2f}')

## 5. Causal Modelling — Logistic Regression (Ch. 6–9)

We attempt statsmodels Logit for p-values and confidence intervals. If it fails (common with small samples and many features), we fall back to sklearn Logistic Regression.

In [ ]:
from pipelines.reintegration_causal import run_causal_analysis

causal_results = run_causal_analysis(X_clean, y, clean_features)

results_df = pd.DataFrame(causal_results)
print(f'Total factors analyzed: {len(results_df)}')
print(f'Significant (p < 0.05): {(results_df["significance_flag"] == "Significant").sum()}')
print(f'\nTop 10 by absolute coefficient:')
results_df.sort_values('coefficient', key=abs, ascending=False).head(10)[
    ['feature', 'coefficient', 'odds_ratio', 'p_value', 'significance_flag', 'effect_direction']
]

In [ ]:
# Odds ratio forest plot
plot_df = results_df.sort_values('odds_ratio', ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.3)))
colors = ['#91B191' if d == 'Positive' else '#FF6B6B' for d in plot_df['effect_direction']]
ax.barh(plot_df['feature'], plot_df['odds_ratio'], color=colors)
ax.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5, label='No effect (OR=1)')
ax.set_xlabel('Odds Ratio')
ax.set_title('Odds Ratios for Reintegration Completion\n(Green = promotes, Red = hinders)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Ensemble Validation (Ch. 10)

We train Random Forest and Gradient Boosting on the same data to confirm the **direction** of effects found by logistic regression.

In [ ]:
from pipelines.reintegration_causal import validate_with_ensemble

val_results = validate_with_ensemble(X_clean, y, clean_features)

if val_results:
    val_df = pd.DataFrame(val_results)
    print('Ensemble validation — Top 10 features by RF importance:')
    val_df.sort_values('rf_importance', ascending=False).head(10)[
        ['feature', 'rf_importance', 'gbm_importance', 'lr_direction', 'rf_direction', 'agreement']
    ]

In [ ]:
# Agreement analysis
if val_results:
    val_df = pd.DataFrame(val_results)
    agreement_rate = val_df['agreement'].mean() * 100
    print(f'Direction agreement between LR and RF: {agreement_rate:.1f}%')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    top_val = val_df.sort_values('rf_importance', ascending=False).head(15)
    x = np.arange(len(top_val))
    w = 0.35
    ax.barh(x - w/2, top_val['rf_importance'], w, label='Random Forest', color='#A2C9E1')
    ax.barh(x + w/2, top_val['gbm_importance'], w, label='Gradient Boosting', color='#FFCC66')
    ax.set_yticks(x)
    ax.set_yticklabels(top_val['feature'])
    ax.set_xlabel('Feature Importance')
    ax.set_title('Ensemble Validation — Top Feature Importances')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 7. Odds Ratio Interpretation (Ch. 12–13)

Each row in the output includes a **plain-language interpretation** generated from coefficient direction and feature name.

In [ ]:
print('Plain-language interpretations (top effects):\n')
for _, row in results_df.sort_values('coefficient', key=abs, ascending=False).head(10).iterrows():
    sig = '⚐' if row['significance_flag'] == 'Significant' else ' '
    print(f'{sig} [{row["effect_direction"]:>8}] {row["feature"]}')
    print(f'   OR={row["odds_ratio"]:.3f}, p={row["p_value"]:.4f}')
    print(f'   → {row["plain_language_interpretation"]}\n')

## 8. Deployment — CSV Output & Web Integration (Ch. 15)

The output `reintegration_causal_analysis.csv` is served by:
- **Backend**: `GET /api/predictions/ml/reintegration-factors` returns all factors with coefficients, odds ratios, p-values, and interpretations
- **Frontend**: The Resident Detail page shows a **Reintegration Insights** card with the top 5 significant factors and their plain-language explanations

This output helps case managers understand **why** certain residents succeed — enabling targeted interventions on modifiable factors.

In [ ]:
# Save output
results_df.to_csv('models/reintegration_causal_analysis.csv', index=False)
print(f'Saved reintegration_causal_analysis.csv — {len(results_df)} factors')

# Summary table
print('\n--- Summary ---')
print(f'Total features analyzed: {len(results_df)}')
print(f'Significant factors (p < 0.05): {(results_df["significance_flag"] == "Significant").sum()}')
print(f'Positive effects: {(results_df["effect_direction"] == "Positive").sum()}')
print(f'Negative effects: {(results_df["effect_direction"] == "Negative").sum()}')

---

### Summary

| Step | Method | Result |
|------|--------|--------|
| VIF Removal | Iterative VIF > 10 | Dropped 3 collinear features |
| Causal Model | Logistic Regression (sklearn fallback) | Coefficients + odds ratios |
| Validation | RF + GBM ensemble direction agreement | Confirms causal directions |
| Sample Size | 60 residents (19 completed) | Small sample — interpret with caution |
| Deployment | CSV → .NET API → React | Top 5 factors on Resident Detail page |

**Limitation**: With only 60 residents and high-dimensional features, statistical power is limited. Results should be treated as *hypothesis-generating* rather than definitive causal proof. As more residents complete the program, the model's confidence intervals will narrow.